<a href="https://colab.research.google.com/github/ella941223-cyber/Programming-Language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [10]:
!pip install -q google-generativeai

In [11]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [12]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [13]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CEUaBeqPvLAKqdYnz0KVSe7QNkBYWbNbfs0-VhwvY8c/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [15]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

Computers **learn from data** to **find patterns** and **make decisions**, **mimicking human thought**.


### 定義 AI 摘要函式

In [16]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [24]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    clean_summary = summary.replace("**", "").replace("#", "").strip()

    try:
        # --- 修改這一段：捨棄 update_cell，改用 append_row ---
        today = datetime.now().strftime('%Y-%m-%d')

        # 直接把 [日期, "AI 摘要", 內容] 當成一行塞進去
        # append_row 會自動找到最下方的空白列，保證不會蓋到前面的成績
        _ws.append_row([today, "AI 摘要", clean_summary], value_input_option='USER_ENTERED')
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [25]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，以下根據您提供的成績列表，為您整理出一個簡單的摘要與常見迷思：

---

### 學生成績摘要與常見迷思整理

#### 一、 成績摘要

根據2026年04月15日的記錄，該學生在三項科目中的表現分別為：
*   **國文：** 85分
*   **數學：** 78分
*   **英文：** 92分

整體分數分佈在78分至92分之間。

#### 二、 成績評量之常見迷思

成績（分數）是學習過程中的一種衡量工具，但圍繞著它常有一些誤解。以下列出常見迷思：

1.  **迷思一：成績代表一個人的全部能力或智商。**
    *   **解釋：** 成績多半是衡量學生在特定科目、特定時間點上對知識點的掌握程度。它無法全面評估一個人的智慧、創造力、解決問題能力、人際交往能力、批判性思維或實踐技能。許多在學業成績上不突出的人，在其他領域展現出卓越的才能。

2.  **迷思二：高分總是代表對知識的深入理解。**
    *   **解釋：** 學生可能透過良好的記憶力或特定應試技巧獲得高分，但對於概念的深層理解、跨領域應用或獨立思考的能力可能仍有發展空間。反之，偶爾的低分也不代表完全不理解，可能受考試形式、壓力等因素影響。

3.  **迷思三：低分意味著學生不夠努力或沒有能力。**
    *   **解釋：** 分數可能受到多種因素影響，包括考試當天的身體狀況、考試形式適應性、學習方法是否得當、個人壓力、教師評分標準，甚至科目本身的興趣差異。低分不應直接被解讀為努力不足或潛力有限，而應作為一個信號，促使我們去探究背後的原因。

4.  **迷思四：成績是衡量學習成果或未來成功的唯一標準。**
    *   **解釋：** 除了成績，學生在課外活動、團隊合作、社會責任感、情緒智商、實踐技能、面對挑戰的韌性等方面的發展同樣重要。這些軟實力往往無法透過分數完全體現，卻對個人成長和未來職業生涯有深遠影響。

5.  **迷思五：單一

定義 Gradio 處理函式

In [26]:
import gradio as gr

def process_grades_and_summary_wrapper(subject, grade):
    """
    Gradio 介面傳入單一科目和成績，轉換為 process_grades_and_summary 所需的格式。
    """
    # process_grades_and_summary 預期 grade_data 是一個列表的列表，例如：[['國文', '90']]
    # Gradio 的 Number 輸入會是浮點數，需要轉換成字串。
    grade_data = [[subject, str(int(grade))]] # 確保成績是整數後轉為字串
    return process_grades_and_summary(grade_data)

with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請選擇科目並輸入成績，點擊『送出』後系統會自動處理。")

    with gr.Row():
        with gr.Column():
            # 下拉選單：請確保 choices 列表完整
            subject_input = gr.Dropdown(
                choices=["國文", "數學", "英文"],
                label="選擇或手動輸入科目",
                allow_custom_value=True  # 允許使用者自己打字
            )
            grade_input = gr.Number(label="成績", value=0)
            submit_button = gr.Button("送出", variant="primary")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    # --- 關鍵修正處 ---
    submit_button.click(
        fn=process_grades_and_summary_wrapper, # 使用轉接函數
        inputs=[subject_input, grade_input],   # 這裡一定要放兩個輸入
        outputs=[sheet_output, summary_output]
    )

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>